# MLOps Sentiment Analysis
**Master AI Engineering — Progetto MLOps**

Questo notebook illustra l'implementazione di un sistema MLOps completo per l'analisi del sentiment sui social media, sviluppato per MachineInnovators Inc.

Il progetto è strutturato in tre fasi:
- **Fase 1**: Implementazione del modello di analisi del sentiment con RoBERTa
- **Fase 2**: Pipeline CI/CD automatizzata con GitHub Actions
- **Fase 3**: Deploy su HuggingFace e sistema di monitoraggio continuo

**Repository GitHub**: [https://github.com/giovannibonisoli/mlops-sentiment-analysis](https://github.com/giovannibonisoli/mlops-sentiment-analysis)

---

## Setup
Installazione delle dipendenze e clone del repository.

In [ ]:
!pip install transformers torch datasets scikit-learn huggingface_hub numpy

In [ ]:
!git clone https://github.com/tuousername/mlops-sentiment-analysis.git
%cd mlops-sentiment-analysis

---
## Fase 1 — Modello di Sentiment Analysis

### Scelte progettuali

La consegna di utilizzare come base il modello `cardiffnlp/twitter-roberta-base-sentiment-latest`, basato sull'architettura RoBERTa.

Il modello classifica i testi in tre classi: **positive**, **neutral**, **negative**.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment-latest'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
classifier = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)

print(f'Modello caricato: {MODEL_NAME}')
print(f'Label disponibili: {model.config.id2label}')

### Test rapido del modello

In [1]:
examples = [
    'I love this product, it is absolutely amazing!',
    'This is terrible, I am very disappointed.',
    'It is okay, nothing special.'
]

for text in examples:
    result = classifier(text)[0]
    print(f'Text: {text}')
    print(f'Prediction: {result["label"].lower()} (confidence: {result["score"]:.2f})')
    print()

NameError: name 'classifier' is not defined

### Caricamento dataset e valutazione baseline

Viene utilizzato il dataset `tweet_eval/sentiment`. La valutazione baseline viene effettuata sull'intero test set.

La **Macro F1** viene usata come metrica principale invece dell'accuracy perché il dataset è sbilanciato, i neutral sono quasi il doppio dei positive. Su dataset sbilanciati l'accuracy può essere fuorviante, mentre la Macro F1 penalizza i modelli che performano male sulle classi minoritarie.

In [ ]:
from datasets import load_dataset
from collections import Counter

dataset = load_dataset('tweet_eval', 'sentiment')
test_data = dataset['test']

LABEL_MAP = {0: 'negative', 1: 'neutral', 2: 'positive'}
distribution = Counter(test_data['label'])
total = sum(distribution.values())

print(f'Dimensione test set: {len(test_data)}')
print('\nDistribuzione test set:')
for label_id, count in sorted(distribution.items()):
    print(f'{LABEL_MAP[label_id]}: {count} ({count/total:.1%})')

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

y_true = [LABEL_MAP[l] for l in test_data['label']]
y_pred = [classifier(t, truncation=True, max_length=512)[0]['label'].lower()
          for t in test_data['text']]

print('=== Valutazione Baseline ===')
print(classification_report(y_true, y_pred))
print(f'Accuracy: {accuracy_score(y_true, y_pred):.4f}')
print(f'Macro F1: {f1_score(y_true, y_pred, average="macro"):.4f}')

---
## Fase 2 — Pipeline CI/CD

### Architettura della pipeline

La pipeline CI/CD è implementata con GitHub Actions in un singolo workflow (`CI_CD.yml`) con due job separati:

**Job 1 — Test** (eseguito ad ogni push/PR):
- Lint con flake8
- Unit test e integration test con pytest

**Job 2 — Train, Validate e Deploy** (eseguito solo da `workflow_dispatch`):
- Fine-tuning del modello su `tweet_eval/sentiment`
- Validazione: confronto Macro F1 con modello in produzione su HuggingFace
- Deploy su HuggingFace Hub solo se il nuovo modello è migliore

Questa separazione garantisce che l'allenamento e il deploy di un modello avvengano esclusivamente a su trigger manuale, e non ad ogni push.

### Allenamento del modello

In [ ]:
import sys
sys.path.insert(0, '.')
from src.train import train

# Fine-tuning con configurazione di default:
# TRAIN_SAMPLES=1000, VALIDATION_SAMPLES=1000, NUM_EPOCHS=3
train()

### Validazione del modello

Prima del deploy, il nuovo modello viene confrontato con quello in produzione su HuggingFace. Il deploy avviene solo se il nuovo modello ha una Macro F1 superiore.

In [ ]:
import os
from src.evaluate import validate

os.environ['HF_REPO']  = 'giovannibonisoli/sentiment-model'
os.environ['HF_TOKEN'] = 'hf_...'

validate()

---
## Fase 3 — Deploy e Monitoraggio Continuo

### Deploy su HuggingFace Hub

In [ ]:
from src.deploy import deploy

os.environ['MODEL_PATH'] = './models/sentiment_model'
os.environ['HF_REPO']    = 'giovannibonisoli/sentiment-model'
os.environ['HF_TOKEN']   = 'hf_...'

deploy()

### Sistema di Monitoraggio

Il sistema di monitoraggio è composto da tre moduli:

- **`monitor.py`**: logga ogni predizione con timestamp, label e confidence in un CSV
- **`drift.py`**: analizza le predizioni e rileva anomalie tramite distribuzione del sentiment, confidence media e PSI (Population Stability Index)
- **`simulate.py`**: simula l'arrivo di nuovi dati dai social media

In produzione `monitor.py` verrebbe alimentato da testi reali provenienti dalle API dei social media monitorati.

#### Simulazione normale — distribuzione stabile

In [ ]:
os.environ['LOG_FILE'] = './monitoring/predictions_log.csv'

from src.monitoring.simulate import simulate

# Simula 100 testi con distribuzione normale
# Popola il log baseline per il confronto successivo
simulate(samples=100)

#### Simulazione drift — crisi reputazionale

Viene simulata una crisi reputazionale sovracampionando la classe negative (70% negative vs 32% baseline). Questo triggera un alert di distribution shift.

In [ ]:
from src.monitoring.simulate import simulate_drift

simulate_drift()

### Analisi del log di monitoraggio

In [ ]:
import csv
from collections import Counter

rows = []
with open('./monitoring/predictions_log.csv', 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

distribution = Counter(r['predicted_label'] for r in rows)
total = sum(distribution.values())
confidences = [float(r['confidence']) for r in rows]

print(f'Totale predizioni loggiate: {len(rows)}')
print('\nDistribuzione complessiva:')
for label, count in distribution.items():
    print(f'  {label}: {count} ({count/total:.1%})')
print(f'\nConfidence media: {sum(confidences)/len(confidences):.4f}')

---
## Conclusioni

Il progetto implementa un sistema MLOps completo per l'analisi del sentiment sui social media:

**Fase 1** — Il modello `cardiffnlp/twitter-roberta-base-sentiment-latest` fornisce una baseline solida con accuracy ~72% e Macro F1 ~0.72 su `tweet_eval/sentiment`.

**Fase 2** — La pipeline CI/CD con GitHub Actions automatizza training, validazione e deploy. Il model validation gate garantisce che venga deployato un modello migliore rispetto a quello in produzione.

**Fase 3** — Il sistema di monitoraggio rileva automaticamente distribution shift e data drift tramite PSI. Ciò rende possibile avviare un retraining del modello in produzione, manualmente.

Il ciclo MLOps completo è:

```
Dati social → Predizione → Log → Monitoraggio → [Drift?] → Retraining → Deploy
```

**Limiti e sviluppi futuri**:
- La pipeline di riallenamento, validazione e deploy è lanciabile solo manualmente.
- I dati per il riallenamento per il momento vengono da dataset opensource
- Il sistema di monitoraggio si basa su dei CSV temporanei.

**Sviluppi futuri**
- Il retraining
- Alimentare il sistema con API reali dei social media
- Il retraining userebbe nuovi dati etichettati raccolti dal monitoraggio
- Il sistema di monitoraggio potrebbe essere esteso con un database persistente.